# Type II Errors and the Power of the Test

Type I errors live at a single point — $\mu = 66.7$, the null value. Type II errors live *everywhere else*: at every $\mu > 66.7$ the alternative allows, there is some probability $\beta$ of failing to reject a false $H_0$. With the advance rule from last notebook — reject iff $\bar{x} > 68.26$ — that probability is a plain computation: if the truth were $\mu = 70$, then $\bar{X} \sim N(70, (3/\sqrt{10})^2)$ and $\beta = P(\bar{X} \le 68.26) = 0.0333$. Flip it around and you get the **power of the test**, $1 - \beta$: the probability of *correctly* rejecting, drawn as a curve over every alternative $\mu$ — starting at height $\alpha$ above the null value and climbing to 1. Bigger $\alpha$ lifts the whole curve (the trade-off again); a bigger sample lifts it for free. It is based on the sixth video of Lesson 8 (Hypothesis Testing).

## 1. Introduction

The significance level has hogged the spotlight since notebook 02; this notebook finally gives the other error its numbers. The key was the last notebook's advance decision rule: with the rejection region frozen at 68.26, "fail to reject when the truth is $\mu$" is just an area under a shifted curve — and sweeping $\mu$ turns that area into the test's full performance profile.

After completing this notebook you will be able to:

- Explain why Type I probability is a single number while Type II probability $\beta(\mu)$ is a *function* over the alternative.
- Compute $\beta$ for any alternative mean — reproducing the video's $\beta(70) = 0.0333$ — and note what it does *not* depend on: the observed data.
- Define the **power** $1 - \beta(\mu)$, read its curve (height $\alpha$ at $\mu_0$, climbing to 1), and connect each region to the outcome table.
- Show how raising $\alpha$ raises power everywhere — the fixed-$n$ trade-off — and how raising $n$ escapes it.
- Design a test: find the sample size that achieves target $\alpha$ *and* target $\beta$ simultaneously.

## 2. Theory

### One point versus a whole half-line

Recall the two errors for the height test ($H_0: \mu = 66.7$, $H_1: \mu > 66.7$):

- A **Type I error** — rejecting a true null — can only happen at **one** value of the population mean: 66.7.
- A **Type II error** — failing to reject a false null — can happen at **any** $\mu > 66.7$. So its probability isn't one number but a function of which alternative is actually true.

### Computing $\beta$ at $\mu = 70$

Design as before: $n = 10$, $\sigma = 3$, $\alpha = 0.05$, and the critical-value rule **reject iff $\bar{x} > 68.26$**. If the true mean were 70, what is the chance of an error? That's the probability of *not* rejecting:

$$
\beta = P(\bar{X} \le 68.26 \mid \mu = 70)
$$

Under this alternative the sample mean no longer follows the null curve: $\bar{X} \sim N(70,\, (3/\sqrt{10})^2)$ — same spread, new center. The area below 68.26 under *that* curve is

$$
\beta(70) = 0.0333
$$

Two remarks the video stresses:

1. $\beta$ **does not depend on the observed sample** — only on the design: the significance level (through the critical value), $n$, $\sigma$, and the alternative $\mu$ you're pricing. (This is exactly why the advance rule was needed.)
2. Nothing is special about 70 — you can compute $\beta(\mu)$ for **as many alternative values as you want**.

### The power of the test

Often you'd rather characterize the chances of a *good* decision — the outcome-table quadrant where you correctly reject a false null. That is the **power of the test**:

$$
\text{power}(\mu) = P(\text{reject } H_0 \mid \mu) = 1 - \beta(\mu) \qquad \text{for each } \mu \text{ in } H_1
$$

### Reading the power curve

For the right-tailed test, the power curve over $\mu$:

- At the far left, $\mu = 66.7$: the height is **exactly $\alpha$** — the probability of rejecting when the null value is true is, by construction, the significance level.
- At $\mu = 68$: the height is the power there — the probability of catching a true increase to 68 — and the gap between the curve and 1 is $\beta(68)$.
- Same reading at $\mu = 70$, where the curve is already close to 1.
- The curve **increases toward 1** as $\mu$ grows: a larger true mean shifts $\bar{X}$'s distribution right, so less and less of it stays below the fixed critical value.

### Three $\alpha$'s, and the escape hatch

Draw the curve for $\alpha = 0.01, 0.05, 0.1$: increasing $\alpha$ raises the power at *every* $\mu$ — and, correspondingly, being too restrictive with Type I errors inflates Type II. **At a fixed sample size the trade-off is inescapable.** But if you can choose any sample size you need, you can drive $\alpha$ *and* $\beta$ both as small as you want — the design story this notebook ends on.

## 3. Worked Examples

Throughout: SE $= 3/\sqrt{10} = 0.949$, critical value $k = 68.26$.

### Example 1: $\beta(70)$ by hand

$z = (68.26 - 70)/0.949 = -1.834$, so $\beta = \Phi(-1.834) = 0.0333$. A true jump to 70 inches would be missed only ~3% of the time — this design is nearly certain to catch so large a change.

### Example 2: power at the video's landmarks

- $\text{power}(68) = 1 - \Phi\big((68.26 - 68)/0.949\big) = 1 - \Phi(0.274) = 0.392$ — a true mean of 68 is caught only ~39% of the time (this is the same 39.6% the simulation in notebook 03 found, and the 0.393 of notebook 04's exercise).
- $\text{power}(70) = 1 - 0.0333 = 0.967$.

The lesson in the pair: power depends dramatically on *how far* the truth is from the null.

### Example 3: why the curve starts at $\alpha$

At $\mu = 66.7$ the "alternative" coincides with the null, and $P(\text{reject} \mid \mu = 66.7)$ is the Type I probability — which the critical value was built to hold at $\alpha = 0.05$. The power curve therefore *continuously extends* the Type I story into the alternative: one curve, with the outcome table's two "reject" cells written along it.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project), then $\beta$ and power as functions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(66)

MU0, SIGMA, N = 66.7, 3.0, 10

def z_quantile(p):
    return float(sp.sqrt(2) * sp.erfinv(2 * sp.Float(p) - 1))

Phi = np.vectorize(lambda z: 0.5 * (1 + float(sp.erf(sp.Float(float(z)) / sp.sqrt(2)))))

def beta(mu, alpha=0.05, n=N):
    """P(fail to reject | true mean mu), right-tailed test with known sigma."""
    se = SIGMA / np.sqrt(n)
    k = MU0 + z_quantile(1 - alpha) * se        # the advance decision rule
    return Phi((k - mu) / se)

def power(mu, alpha=0.05, n=N):
    return 1 - beta(mu, alpha, n)

### The video's numbers — and a simulation witness

In [ ]:
print(f"critical value: {MU0 + z_quantile(0.95) * SIGMA / np.sqrt(N):.2f}")
print(f"beta(70)  = {float(beta(70)):.4f}   (the video's 0.0333)")
print(f"power(68) = {float(power(68)):.3f}   (notebook 03's simulated 39.6%)")
print(f"power(70) = {float(power(70)):.3f}")

sim = rng.normal(70, SIGMA / np.sqrt(N), size=200_000)
print(f"\nsimulated beta(70): {(sim <= 68.26).mean():.4f} -- an area under a")
print("shifted curve, nothing more. And note: no observed sample appears anywhere")
print("in the computation; beta is a property of the DESIGN, not the data.")

### $\beta$ as a function: price any alternative you like

In [ ]:
print(f"{'true mu':>8s} {'beta':>8s} {'power':>8s}")
for mu in (67.0, 67.5, 68.0, 68.5, 69.0, 70.0, 71.0):
    print(f"{mu:>8.1f} {float(beta(mu)):>8.3f} {float(power(mu)):>8.3f}")
print("\nClose alternatives are almost invisible (beta near 1 - alpha); distant")
print("ones are almost certain catches. Power is a curve, not a number.")

## 5. Visualizations

### The picture behind $\beta(70)$

Two curves, one threshold: the null curve fixed the critical value; the alternative curve pays the price — its blue area below 68.26 is $\beta$.

In [ ]:
se = SIGMA / np.sqrt(N)
k = MU0 + z_quantile(0.95) * se
x_axis = np.linspace(63, 74, 800)
f_null = np.exp(-0.5 * ((x_axis - MU0) / se) ** 2) / (se * np.sqrt(2 * np.pi))
f_alt = np.exp(-0.5 * ((x_axis - 70) / se) ** 2) / (se * np.sqrt(2 * np.pi))

plt.figure(figsize=(10.5, 4.8))
plt.plot(x_axis, f_null, color="tab:green", linewidth=1.8,
         label=r"under $H_0$: $N(66.7, SE^2)$")
plt.plot(x_axis, f_alt, color="tab:blue", linewidth=1.8,
         label=r"if truth is $\mu = 70$: $N(70, SE^2)$")
plt.fill_between(x_axis, f_null, where=x_axis >= k, color="tab:green", alpha=0.4)
plt.fill_between(x_axis, f_alt, where=x_axis <= k, color="tab:blue", alpha=0.4)
plt.axvline(k, color="black", linestyle="--", linewidth=1.5)
plt.annotate(r"$\alpha$ = 0.05", xy=(68.8, 0.035), fontsize=10, color="tab:green")
plt.annotate(r"$\beta(70)$ = 0.0333", xy=(66.9, 0.035), fontsize=10, color="tab:blue")
plt.annotate(f"k = {k:.2f}", xy=(k, 0.43), ha="center", fontsize=10)
plt.xlabel(r"$\bar{x}$ (inches)")
plt.ylabel("density")
plt.title("The same threshold cuts both worlds: green tail = Type I, blue belly = Type II")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The power curve

Height $\alpha$ at the null value, then a climb toward certainty — with the video's landmark readings at 68 and 70.

In [ ]:
mu_axis = np.linspace(66.7, 71.5, 300)
power_curve = power(mu_axis)

plt.figure(figsize=(10, 5))
plt.plot(mu_axis, power_curve, color="tab:purple", linewidth=2.5)
plt.axhline(1, color="gray", linewidth=0.8)
plt.axhline(0.05, color="tab:green", linestyle=":", linewidth=1.2)
plt.annotate(r"height at $\mu_0$ = exactly $\alpha$ = 0.05", xy=(66.75, 0.09),
             fontsize=9, color="tab:green")
for mu_mark in (68.0, 70.0):
    p_mark = float(power(mu_mark))
    plt.plot([mu_mark, mu_mark], [0, p_mark], color="gray", linestyle=":", linewidth=1)
    plt.plot(mu_mark, p_mark, "ko", markersize=5)
    plt.annotate(f"power({mu_mark:.0f}) = {p_mark:.3f}", xy=(mu_mark, p_mark),
                 xytext=(8, -12), textcoords="offset points", fontsize=9)
    plt.annotate(f"$\\beta$ = {1 - p_mark:.3f}", xy=(mu_mark, (1 + p_mark) / 2),
                 xytext=(8, 0), textcoords="offset points", fontsize=9, color="tab:blue")
    plt.plot([mu_mark, mu_mark], [p_mark, 1], color="tab:blue", linewidth=2.5, alpha=0.6)
plt.xlabel(r"true population mean $\mu$ (the alternatives)")
plt.ylabel(r"power = P(reject $H_0$ | $\mu$)")
plt.title("The power of the test: the probability of catching each possible truth")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.show()

### Three $\alpha$'s: the trade-off as three curves

A looser $\alpha$ lifts power everywhere; a stricter one drops it — Type II pays for Type I, point by point.

In [ ]:
plt.figure(figsize=(10, 5))
for alpha, color in [(0.10, "tab:orange"), (0.05, "tab:purple"), (0.01, "tab:red")]:
    plt.plot(mu_axis, power(mu_axis, alpha=alpha), color=color, linewidth=2,
             label=f"$\\alpha$ = {alpha}: power(70) = {float(power(70, alpha=alpha)):.3f}")
    plt.plot(66.7, alpha, "o", color=color, markersize=5)
plt.axvline(70, color="gray", linestyle=":", linewidth=1)
plt.xlabel(r"true population mean $\mu$")
plt.ylabel("power")
plt.title("Raising alpha raises power everywhere -- the fixed-n trade-off, drawn")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The escape hatch: sample size

Same $\alpha = 0.05$, growing $n$: the curve steepens toward a step function — high power against even nearby alternatives, with the Type I rate untouched.

In [ ]:
plt.figure(figsize=(10, 5))
for n, color in [(10, "tab:purple"), (30, "tab:blue"), (100, "tab:green")]:
    plt.plot(mu_axis, power(mu_axis, n=n), color=color, linewidth=2,
             label=f"n = {n}: power(68) = {float(power(68, n=n)):.3f}")
plt.axvline(68, color="gray", linestyle=":", linewidth=1)
plt.axhline(0.05, color="black", linestyle=":", linewidth=0.8)
plt.xlabel(r"true population mean $\mu$")
plt.ylabel("power")
plt.title("Fixed alpha = 0.05: more data buys power against every alternative")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The whole story on one page

Everything above in a single sheet: one threshold in two worlds, the power curve, the β table, and how α and n move the curve.

<img src="img/06_power.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Compute $\beta(69)$ and the power at 69 for the standard design ($\alpha = 0.05$, $n = 10$). Which outcome-table cell does each number price?

**Exercise 2.** Explain in two sentences why $\beta$ is a function while $\alpha$ is a single number — and what extra ingredient (beyond the null) a $\beta$ computation always needs.

### Intermediate

**Exercise 3.** Tabulate $\beta(\mu)$ for $\mu = 67, 68, 69, 70$ and explain the monotone pattern via the picture of the shifting alternative curve against the fixed critical value.

**Exercise 4.** At $\mu = 68$, compare the power for $\alpha = 0.01, 0.05, 0.10$. Then state the video's two-sided moral: what does tightening $\alpha$ do to $\beta$, and what single design change breaks the trade-off?

### Advanced

**Exercise 5.** *Designing for both errors.* Find the smallest $n$ such that the right-tailed test has $\alpha = 0.05$ *and* $\beta(68) \le 0.05$. Derive it from $\mu_0 + z_{0.95}\sigma/\sqrt{n} \le 68 - z_{0.95}\sigma/\sqrt{n}$, then verify with the `power` function and a simulation.

**Exercise 6.** *The two-tailed price.* At the same $\alpha = 0.05$ and $n = 10$, plot the power curves of the right-tailed and two-tailed tests across $\mu \in [64, 71.5]$. Where does each dominate, and how does this complete the story from notebook 03's exercise about choosing the tail from the question?

## 7. Solutions

In [ ]:
# Exercise 1: mu = 69
b69, p69 = float(beta(69)), float(power(69))
print(f"beta(69) = {b69:.3f}  -- prices the 'fail to reject, H1 true' cell (Type II)")
print(f"power(69) = {p69:.3f} -- prices the 'reject, H1 true' cell (correct catch)")

In [ ]:
# Exercise 2: why a function
print("Alpha is evaluated at the single mean the null names (66.7), so it is one")
print("number. A Type II error can occur at every mean the alternative allows, so")
print("beta needs an extra ingredient no null provides: WHICH alternative is true.")
print("Priced across all of them, beta becomes a function of mu.")

In [ ]:
# Exercise 3: the beta table
print(f"{'mu':>6s} {'beta':>8s}")
for mu in (67.0, 68.0, 69.0, 70.0):
    print(f"{mu:>6.1f} {float(beta(mu)):>8.3f}")
print("\nAs mu grows, the alternative curve slides right while the critical value")
print("stays at 68.26 -- ever less of the curve remains below the threshold, so")
print("beta falls monotonically from nearly 1 - alpha toward 0.")

In [ ]:
# Exercise 4: alpha comparison at mu = 68
for a in (0.01, 0.05, 0.10):
    print(f"alpha = {a:>4}: power(68) = {float(power(68, alpha=a)):.3f}, "
          f"beta(68) = {float(beta(68, alpha=a)):.3f}")
print("\nTightening alpha starves power and feeds beta -- at fixed n the two")
print("errors trade off. The one change that improves both: a larger sample.")

In [ ]:
# Exercise 5: n for alpha = 0.05 AND beta(68) <= 0.05
z95 = z_quantile(0.95)
n_exact = (2 * z95 * SIGMA / (68 - MU0)) ** 2
n_star = int(np.ceil(n_exact))
print(f"condition: 2 * {z95:.3f} * 3 / sqrt(n) <= 1.3  ->  n >= {n_exact:.1f} "
      f"->  n = {n_star}")
print(f"check: beta(68) at n = {n_star}: {float(beta(68, n=n_star)):.4f}")

se_star = SIGMA / np.sqrt(n_star)
k_star = MU0 + z95 * se_star
sim5 = rng.normal(68, se_star, size=200_000)
print(f"simulated beta: {(sim5 <= k_star).mean():.4f}")
print(f"\nWith {n_star} people instead of 10, the test holds alpha at 5% AND misses")
print("a true mean of 68 at most 5% of the time -- both dials small at once.")

In [ ]:
# Exercise 6: right-tailed vs two-tailed power
def power_two(mu, alpha=0.05, n=N):
    se_ = SIGMA / np.sqrt(n)
    z_ = z_quantile(1 - alpha / 2)
    lo_, hi_ = MU0 - z_ * se_, MU0 + z_ * se_
    return (1 - Phi((hi_ - mu) / se_)) + Phi((lo_ - mu) / se_)

mu_wide = np.linspace(64, 71.5, 400)
plt.figure(figsize=(10, 4.8))
plt.plot(mu_wide, power(mu_wide), color="tab:purple", linewidth=2,
         label="right-tailed power")
plt.plot(mu_wide, power_two(mu_wide), color="tab:orange", linewidth=2,
         label="two-tailed power")
plt.axvline(MU0, color="black", linestyle=":", linewidth=1)
plt.xlabel(r"true population mean $\mu$")
plt.ylabel("power")
plt.title("Exercise 6: the right-tailed test is stronger where it looks, blind where it doesn't")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

print(f"at mu = 68:  right {float(power(68)):.3f} vs two-tailed {float(power_two(68)):.3f}")
print(f"at mu = 65:  right {float(power(65)):.4f} vs two-tailed {float(power_two(65)):.3f}")
print("\nRight-tailed spends its whole alpha budget on increases: more power there,")
print("essentially none against decreases. The honest chooser picks the tail from")
print("the question -- and this curve is the cost sheet of that choice.")

## 8. Conclusion

- Type I probability lives at the single null value; **Type II probability $\beta(\mu)$ is a function** over every alternative, computable one $\mu$ at a time — and depending only on the design, never the observed data.
- The video's computation: with the rule "reject iff $\bar{x} > 68.26$", a true mean of 70 shifts $\bar{X}$ to $N(70, SE^2)$ and leaves $\beta(70) = 0.0333$ below the threshold.
- **Power** $= 1 - \beta(\mu) = P(\text{reject} \mid \mu)$: the good-decision curve, starting at exactly $\alpha$ over the null value and climbing to 1 — power(68) $= 0.392$, power(70) $= 0.967$ here.
- Raising $\alpha$ lifts the whole power curve (and shrinks $\beta$) — the fixed-$n$ trade-off in one picture; but **with a free choice of sample size, $\alpha$ and $\beta$ can both be made as small as you want** ($n = 58$ already gives $\alpha = \beta(68) = 0.05$).
- Bonus from the exercises: the right-tailed test out-powers the two-tailed one against increases and is blind to decreases — the quantified cost sheet of choosing a tail.